## Introduction

This notebook shows how to build a neural network with NNVM python frontend and generate runtime library for Raspberry Pi with TVM. To run this notebook, you need to install tvm and nnvm following [these instructions](https://github.com/dmlc/nnvm/blob/master/docs/how_to/install.md). Notice that you need to build tvm with llvm.

In [ ]:
import tvm
import nnvm.compiler
import nnvm.testing

## Define Neural Network in nnvm

First, let's define a neural network with nnvm python frontend. For simplicity, we'll use pre-defined resnet-18 network in nnvm. Parameters are initialized with Xavier initializer. In this tutorial, we assume we will do inference on our device and the batch size is set to be 1. Input images are RGB color images of size 224 * 224.

In [ ]:
batch_size = 1
image_shape = (3, 224, 224)

data_shape = (batch_size,) + image_shape

net, params = nnvm.testing.resnet.get_workload(batch_size=batch_size, image_shape=image_shape)
print(net.debug_str())

## Compile with tvm

Next step is to compile model with tvm. Here we set optimization level to be 0 to indicate "SimplifyInference", which will minimize the size of built library.

In [ ]:
opt_level = 0
with nnvm.compiler.build_config(opt_level=opt_level):
    graph, lib, params = nnvm.compiler.build(
        net, tvm.target.rasp(), shape={"data": data_shape}, params=params)

lib_fname = 'net.o'
lib.save(lib_fname)

A net.o library file will be created under the current folder with size 225KB.